# Day 2 — Synthetic control counterfactual

**Goal:** Build a weighted donor combination that tracks the treated unit pre-tariff, then read off the post-tariff gap as the effect estimate.

1. Why synthetic control beats naive before/after for a **single treated unit**
2. Inject a **known** peak-load reduction into a Pecan-shaped simulated panel
3. Fit simplex weights on the pre-period and plot treated vs synthetic counterfactual

See also: [`docs/causal_mental_model.md`](../docs/causal_mental_model.md)

In [ ]:
import matplotlib.pyplot as plt

from src.causal.simulate import SynthSimulationConfig, simulate_heterogeneous_load
from src.causal.synth import (
    daily_peak_series,
    fit_synthetic_control,
    in_space_placebos,
    inject_peak_reduction,
    naive_before_after_series,
)
from src.data.pecan_street import PecanStreetNotAvailableError, load_pecan_street

## Mental model

- **Treated unit:** mean evening peak load of a small pilot group (5 Pecan-style homes).
- **Donors:** untreated households with similar pre-period load paths.
- **Weights:** fit on pre-tariff days so `synthetic ≈ treated` before rollout.
- **Counterfactual:** same weights applied post-tariff estimate `Y(0)`.
- **Effect:** post-period gap `actual − synthetic`.

Naive before/after confounds the tariff with shared shocks (weather). Placebo tests ask whether a fake treated unit would show a gap this large.

## Set up the tariff experiment

Try Pecan Street Dataport; fall back to a simulated heterogeneous panel with a **known 15% peak-load cut** after day 60.

In [ ]:
REDUCTION_PCT = 0.15
config = SynthSimulationConfig(seed=7)
intervention_day = config.n_days_before

try:
    raw = load_pecan_street()
    raise NotImplementedError("Real Pecan pipeline not wired yet; using simulator.")
except PecanStreetNotAvailableError:
    print("Pecan Street unavailable — using simulated heterogeneous panel.")
    panel = simulate_heterogeneous_load(config)

treated_ids = [f"PS{i:04d}" for i in range(config.n_treated)]
donor_ids = [f"PS{i + 100:04d}" for i in range(config.n_donors)]

panel = inject_peak_reduction(
    panel,
    treated_ids=treated_ids,
    intervention_day=intervention_day,
    reduction_pct=REDUCTION_PCT,
    peak_hours=config.peak_hours,
)

treated, donors = daily_peak_series(
    panel,
    treated_ids=treated_ids,
    donor_ids=donor_ids,
    peak_hours=config.peak_hours,
)
print(f"Daily peak series: {len(treated)} days, {donors.shape[1]} donors")

## Fit weights and compare to ground truth

In [ ]:
result = fit_synthetic_control(treated, donors, intervention_day=intervention_day)
naive = naive_before_after_series(treated, intervention_day=intervention_day)
placebo = in_space_placebos(treated, donors, intervention_day=intervention_day)

top_weights = result.weights[result.weights > 0.01].sort_values(ascending=False)
print("Top donor weights:")
print(top_weights.to_string())
print(f"\nPre-period RMSPE: {result.pre_rmspe:.4f} kW")
print(f"Synthetic control ATT: {result.att_kw:.3f} kW ({result.att_pct:.1%} vs synthetic)")
print(f"Injected reduction: {-REDUCTION_PCT:.1%}")
print(f"Naive before/after gap: {naive:.3f} kW (confounded by weather jump)")
print(f"In-space placebo p-value: {placebo.p_value:.3f}")

## Counterfactual plot

Treated actual vs synthetic counterfactual around the intervention date. The post-date gap is the effect estimate.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
days = result.treated.index

ax.plot(days, result.treated, label="Treated actual", color="C0", linewidth=2)
ax.plot(days, result.synthetic, label="Synthetic counterfactual", color="C1", linewidth=2, linestyle="--")
ax.axvline(intervention_day, color="black", linestyle=":", linewidth=1.5, label="Tariff start")
ax.axvspan(intervention_day, days.max(), color="C0", alpha=0.08)

post_gap = result.att_kw
ax.annotate(
    f"Post gap ≈ {post_gap:.2f} kW\n({result.att_pct:.1%})",
    xy=(intervention_day + 5, result.treated.loc[result.treated.index >= intervention_day].mean()),
    xytext=(intervention_day + 12, result.treated.max() - 0.1),
    arrowprops={"arrowstyle": "->", "color": "0.3"},
    fontsize=10,
)

ax.set_xlabel("Day")
ax.set_ylabel("Mean peak-hour load (kW)")
ax.set_title("Synthetic control: treated vs counterfactual peak load")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()